# Tutorial 3: Rho Controller

A self-contained walkthrough of the rho-adaptive controller on a small synthetic classification problem.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from ghosts.control import RhoScaledAdam
from ghosts.radii import compute_rho_out

model = nn.Linear(8, 4)
opt = RhoScaledAdam(model.parameters(), lr=0.01, rhoCap=1.0)

x = torch.randn(4, 8)
targets = torch.tensor([0, 1, 2, 3])

logits = model(x)
loss = nn.functional.cross_entropy(logits, targets)
opt.zero_grad()
loss.backward()
opt.step(logits=logits)

stats = opt.getStats()
print(f"rho = {stats['rho']:.4f}")
print(f"base LR = {stats['baseLR']}")
print(f"effective LR = {stats['effectiveLR']:.6f}")
print(f"scale = {stats['scale']:.4f}")


In [ ]:
torch.manual_seed(42)
n_samples, n_features, n_classes = 64, 16, 5
X = torch.randn(n_samples, n_features)
Y = torch.randint(0, n_classes, (n_samples,))

def train_loop(opt_class, lr, steps=120, **kwargs):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(n_features, 32), nn.ReLU(), nn.Linear(32, n_classes))
    if opt_class == RhoScaledAdam:
        opt = opt_class(net.parameters(), lr=lr, **kwargs)
    else:
        opt = opt_class(net.parameters(), lr=lr)
    losses, rhos, eff_lrs = [], [], []
    for _ in range(steps):
        logits = net(X)
        loss = nn.functional.cross_entropy(logits, Y)
        opt.zero_grad()
        loss.backward()
        if opt_class == RhoScaledAdam:
            opt.step(logits=logits)
            s = opt.getStats()
            rhos.append(s['rho'])
            eff_lrs.append(s['effectiveLR'])
        else:
            opt.step()
            rho = compute_rho_out(logits.detach(), gap='maxmin', reduce='mean')
            rhos.append(rho)
            eff_lrs.append(lr)
        losses.append(loss.item())
    return losses, rhos, eff_lrs


In [ ]:
lr = 0.01
l_adam, r_adam, e_adam = train_loop(torch.optim.Adam, lr)
l_rho, r_rho, e_rho = train_loop(RhoScaledAdam, lr, rhoCap=1.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(l_adam, label='Adam')
axes[0].plot(l_rho, label=r'$\rho$-Adam')
axes[0].set_ylabel('Loss')
axes[0].set_xlabel('Step')
axes[0].legend()
axes[0].set_title('Training loss')

axes[1].plot(r_adam, label='Adam')
axes[1].plot(r_rho, label=r'$\rho$-Adam')
axes[1].set_ylabel(r'$\rho$')
axes[1].set_xlabel('Step')
axes[1].legend()
axes[1].set_title('Convergence radius')

axes[2].plot(e_adam, label='Adam')
axes[2].plot(e_rho, label=r'$\rho$-Adam')
axes[2].set_ylabel('Effective LR')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].set_title('Effective learning rate')

plt.tight_layout()
plt.show()


In [ ]:
configs = [
    {'rhoCap': 1.0, 'rhoFloor': 0.01, 'label': 'cap=1.0, floor=0.01'},
    {'rhoCap': 0.5, 'rhoFloor': 0.01, 'label': 'cap=0.5, floor=0.01'},
    {'rhoCap': 1.0, 'rhoFloor': 0.1, 'label': 'cap=1.0, floor=0.1'},
]

fig, ax = plt.subplots(figsize=(6, 4))
for cfg in configs:
    losses, _, _ = train_loop(
        RhoScaledAdam,
        0.01,
        rhoCap=cfg['rhoCap'],
        rhoFloor=cfg['rhoFloor'],
    )
    ax.plot(losses, label=cfg['label'])

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title(r'Effect of $\rho$ cap and floor')
ax.legend()
plt.tight_layout()
plt.show()
